# Rapport France – Interconnexions électriques
Génère le rapport complet (flux physiques & monétaires, rentes de congestion, taux d'utilisation) à partir des fichiers Parquet produits par `load_explicit.ipynb`.

## 0. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import math
import plotly.graph_objects as go
from IPython.display import display

YEARS = list(range(2014, 2026))

PRICES_PATH          = "data/prices.parquet"
FLOWS_PATH           = "data/flows.parquet"
DAILY_CAPACITY_PATH  = "data/daily_capacity.parquet"
HOURLY_REPORT_PATH   = "hourly_report.parquet"

## 1. Chargement des données

In [ ]:
prices         = pd.read_parquet(PRICES_PATH)
flows          = pd.read_parquet(FLOWS_PATH)
daily_capacity = pd.read_parquet(DAILY_CAPACITY_PATH)

print(f"Prix        : {len(prices):,} lignes  —  pays : {sorted(prices['country'].unique())}")
print(f"Flux        : {len(flows):,} lignes  —  partenaires : {sorted(flows['partner'].dropna().unique())}")
print(f"Cap. journ. : {len(daily_capacity):,} lignes")

# ── Correction FX : prix Grande-Bretagne GBP → EUR ───────────────────────────
# Taux EUR/GBP annuels moyens (source : BCE / ECB Statistical Data Warehouse)
# Prix GB dans ENTSO-E sont en GBP ; on convertit en EUR pour homogénéiser les spreads.
EUR_GBP = {
    2014: 0.8063, 2015: 0.7258, 2016: 0.8189, 2017: 0.8764,
    2018: 0.8847, 2019: 0.8778, 2020: 0.8897, 2021: 0.8602,
    2022: 0.8529, 2023: 0.8693, 2024: 0.8452, 2025: 0.8350,
}

gb_mask  = prices["country"] == "Great Britain"
gb_years = pd.to_datetime(prices.loc[gb_mask, "datetime"]).dt.year
prices.loc[gb_mask, "price"] = prices.loc[gb_mask, "price"] / gb_years.map(EUR_GBP)

print(f"\nCorrection FX appliquée sur {gb_mask.sum():,} lignes GB (GBP → EUR)")
print(prices[gb_mask]["price"].describe().round(2))

## 2. Construction des rapports

In [ ]:
def compute_monetary_flows(flows, prices):
    df = flows.copy()
    df = df.merge(
        prices.rename(columns={"country": "from_country", "price": "price_export"}),
        on=["from_country", "datetime"], how="left"
    )
    df = df.merge(
        prices.rename(columns={"country": "to_country", "price": "price_import"}),
        on=["to_country", "datetime"], how="left"
    )
    df["value_eur"]       = df["flow_mw"] * df["price_export"]
    df["congestion_rent"] = df["flow_mw"] * (df["price_import"] - df["price_export"])
    return df


def fill_hourly_with_daily(hourly_report, daily_capacity):
    df = hourly_report.copy()
    df["day"]         = pd.to_datetime(df["datetime"]).dt.normalize()
    df["capacity_mw"] = df["capacity_mw"].replace(0, np.nan)

    # dédupliquer les capacités journalières au cas où le parquet contient des doublons
    daily_dedup = (
        daily_capacity
        .groupby(["day", "from_country", "to_country"], as_index=False)
        ["capacity_mw"].sum()
    )

    df = df.merge(
        daily_dedup,
        on=["day", "from_country", "to_country"],
        how="left", suffixes=("", "_daily")
    )
    df["capacity_mw"] = df["capacity_mw"].fillna(df["capacity_mw_daily"])
    return df.drop(columns=["day", "capacity_mw_daily"])


def merge_hourly_report(flows, prices, daily_capacities):
    flows_money = compute_monetary_flows(flows, prices)
    flows_money["capacity_mw"] = np.nan  # sera remplie par les capacités journalières
    hourly_report = fill_hourly_with_daily(flows_money, daily_capacities)
    hourly_report["utilization_rate"] = np.where(
        hourly_report["capacity_mw"].notna(),
        hourly_report["flow_mw"] / hourly_report["capacity_mw"],
        np.nan
    )
    return hourly_report

In [ ]:
def aggregate_yearly(df):
    def compute_stats(g):
        exp = g[g.from_country == "France"]
        imp = g[g.to_country   == "France"]

        export_flow  = exp["flow_mw"].sum()
        import_flow  = imp["flow_mw"].sum()
        export_value = exp["value_eur"].sum()
        import_value = imp["value_eur"].sum()

        # Rente signée : positive = rente économiquement réalisée
        #                négative = flux à contre-prix (coût d'inefficience)
        export_rent_signed = exp["congestion_rent"].sum()
        import_rent_signed = imp["congestion_rent"].sum()

        # Rente "absolue" : somme des |rentes| par heure (revenu théorique GRT)
        export_rent_abs = exp["congestion_rent"].abs().sum()
        import_rent_abs = imp["congestion_rent"].abs().sum()

        total_flow      = export_flow + import_flow
        total_rent_abs  = export_rent_abs + import_rent_abs

        return pd.Series({
            "export_twh":                      export_flow  / 1e6,
            "import_twh":                      import_flow  / 1e6,
            "export_value_M€":                 export_value / 1e6,
            "import_value_M€":                 import_value / 1e6,
            "export_price_avg_€/MWh":          (export_value / export_flow) if export_flow > 0 else np.nan,
            "import_price_avg_€/MWh":          (import_value / import_flow) if import_flow > 0 else np.nan,
            # Rentes signées (nouveau)
            "export_signed_congestion_rent_M€": export_rent_signed / 1e6,
            "import_signed_congestion_rent_M€": import_rent_signed / 1e6,
            # Rentes absolues (existant)
            "export_abs_congestion_rent_M€":   export_rent_abs / 1e6,
            "import_abs_congestion_rent_M€":   import_rent_abs / 1e6,
            "export_capacity_twh":             exp["capacity_mw"].sum() / 1e6,
            "import_capacity_twh":             imp["capacity_mw"].sum() / 1e6,
            # FIX : on utilise abs() pour éviter de moyenner des taux négatifs
            "export_utilization_rate":         exp["utilization_rate"].abs().mean(),
            "import_utilization_rate":         imp["utilization_rate"].abs().mean(),
            "utilization_rate":                g["utilization_rate"].abs().mean(),
            "spread_avg_€/MWh":                (total_rent_abs / total_flow) if total_flow > 0 else np.nan,
            "abs_congestion_rent_M€":          total_rent_abs / 1e6,
        })

    return (
        df.groupby(["year", "partner"])
        .apply(compute_stats, include_groups=False)
        .reset_index()
        .rename(columns={"partner": "country"})
    )


# ── Agrégation mensuelle & saisonnière ────────────────────────────────────────

SEASONS = {
    12: "Hiver", 1: "Hiver",  2: "Hiver",
     3: "Printemps", 4: "Printemps", 5: "Printemps",
     6: "Été",  7: "Été",   8: "Été",
     9: "Automne", 10: "Automne", 11: "Automne",
}

def aggregate_monthly(df):
    d = df.copy()
    d["month"]  = pd.to_datetime(d["datetime"]).dt.month
    d["season"] = d["month"].map(SEASONS)

    def stats(g):
        exp = g[g.from_country == "France"]
        imp = g[g.to_country   == "France"]
        export_flow = exp["flow_mw"].sum()
        import_flow = imp["flow_mw"].sum()
        total_flow  = export_flow + import_flow
        export_rent = exp["congestion_rent"].abs().sum()
        import_rent = imp["congestion_rent"].abs().sum()
        total_rent  = export_rent + import_rent
        return pd.Series({
            "export_twh":             export_flow / 1e6,
            "import_twh":             import_flow / 1e6,
            "net_twh":               (export_flow - import_flow) / 1e6,
            # FIX : abs() sur utilization_rate
            "utilization_rate":       g["utilization_rate"].abs().mean(),
            "spread_avg_€/MWh":      (total_rent / total_flow) if total_flow > 0 else np.nan,
            "abs_congestion_rent_M€": total_rent / 1e6,
        })

    return (
        d.groupby(["year", "month", "season", "partner"])
        .apply(stats, include_groups=False)
        .reset_index()
        .rename(columns={"partner": "country"})
    )

In [ ]:
hourly_report  = merge_hourly_report(flows, prices, daily_capacity)
yearly_report  = aggregate_yearly(hourly_report)
monthly_report = aggregate_monthly(hourly_report)

print(f"Rapport horaire  : {len(hourly_report):,} lignes")
print(f"Rapport annuel   : {len(yearly_report):,} lignes")
print(f"Rapport mensuel  : {len(monthly_report):,} lignes")
yearly_report.head(10)

In [ ]:
hourly_report.to_parquet("hourly_report.parquet", index=False)
yearly_report.to_csv("yearly_report.csv", index=False)
print("Sauvegardé : hourly_report.parquet  +  yearly_report.csv")

## 3. Fonctions de visualisation

In [ ]:
def histogram_hours(df, year=None):
    if year is not None:
        df = df[df["year"] == year].copy()
    else:
        df = df.copy()

    df["utilization_rate"] = df["flow_mw"].abs() / df["capacity_mw"].abs()
    df["high_utilization"] = df["utilization_rate"] > 0.8
    df["flow_direction"]   = df["flow_mw"].apply(lambda x: "export" if x > 0 else "import")

    counts = (
        df[df["high_utilization"]]
        .groupby(["from_country", "to_country", "flow_direction"])
        .size()
        .reset_index(name="hours_high_utilization")
    )

    fig = go.Figure()
    for direction, color in [("export", "green"), ("import", "red")]:
        sub = counts[counts["flow_direction"] == direction]
        fig.add_trace(go.Bar(
            x=sub.apply(lambda r: f"{r['from_country']} → {r['to_country']}", axis=1),
            y=sub["hours_high_utilization"],
            name=direction.capitalize(),
            marker_color=color
        ))
    fig.update_layout(
        title=f"Heures avec taux d'utilisation > 80 %" + (f" ({year})" if year else ""),
        xaxis_title="Interconnexion",
        yaxis_title="Nombre d'heures",
        barmode="stack"
    )
    return fig

In [ ]:
def histogram_congestion_rent(df, year=None):
    if year is not None:
        df = df[df["year"] == year].copy()

    df = df[
        (df["export_abs_congestion_rent_M€"] > 0) |
        (df["import_abs_congestion_rent_M€"] > 0)
    ].copy()

    fig = go.Figure()
    for col, color, label in [
        ("export_abs_congestion_rent_M€", "green", "Export"),
        ("import_abs_congestion_rent_M€", "red",   "Import"),
    ]:
        sub = df[df[col] > 0]
        fig.add_trace(go.Bar(
            x=sub["country"], y=sub[col],
            name=label, marker_color=color
        ))
    fig.update_layout(
        title="Rente de congestion par partenaire" + (f" ({year})" if year else ""),
        xaxis_title="Pays partenaire",
        yaxis_title="Rente de congestion (M€)",
        barmode="stack"
    )
    return fig

In [ ]:
def plot_congestion_map(yearly_report, year):
    df = yearly_report[yearly_report["year"] == year].copy()
    df["direction"] = np.where(
        df["export_twh"] > df["import_twh"],
        "France exporte", "France importe"
    )
    colors = {"France exporte": "green", "France importe": "red"}

    util_thresh   = df["utilization_rate"].median()
    spread_thresh = df["spread_avg_€/MWh"].median()
    xmax = 1
    ymax = df["spread_avg_€/MWh"].max() * 1.2

    fig = go.Figure()
    for direction, g in df.groupby("direction"):
        fig.add_trace(go.Scatter(
            x=g["utilization_rate"], y=g["spread_avg_€/MWh"],
            mode="markers+text", name=direction,
            text=g["country"], textposition="top center",
            marker=dict(size=12, color=colors[direction], opacity=0.85)
        ))

    fig.update_layout(
        shapes=[
            dict(type="rect", x0=0,           x1=util_thresh, y0=0,            y1=spread_thresh,
                 fillcolor="lightgreen", opacity=0.15, line_width=0, layer="below"),
            dict(type="rect", x0=util_thresh,  x1=xmax,        y0=0,            y1=spread_thresh,
                 fillcolor="lightblue",  opacity=0.15, line_width=0, layer="below"),
            dict(type="rect", x0=0,           x1=util_thresh, y0=spread_thresh, y1=ymax,
                 fillcolor="orange",     opacity=0.15, line_width=0, layer="below"),
            dict(type="rect", x0=util_thresh,  x1=xmax,        y0=spread_thresh, y1=ymax,
                 fillcolor="red",        opacity=0.15, line_width=0, layer="below"),
        ]
    )
    fig.add_vline(x=util_thresh,   line_dash="dash")
    fig.add_hline(y=spread_thresh, line_dash="dash")

    for text, ax, ay in [
        ("Marché intégré",        util_thresh/2,            spread_thresh/2),
        ("Arbitrage efficace",    (util_thresh+xmax)/2,     spread_thresh/2),
        ("Barrière marché",       util_thresh/2,            (spread_thresh+ymax)/2),
        ("Congestion structurelle",(util_thresh+xmax)/2,    (spread_thresh+ymax)/2),
    ]:
        fig.add_annotation(x=ax, y=ay, text=text, showarrow=False)

    fig.update_layout(
        title=f"Carte de congestion France ({year})",
        xaxis_title="Taux d'utilisation moyen horaire",
        yaxis_title="Spread de prix moyen horaire (€/MWh)",
        legend_title="Sens dominant",
        xaxis=dict(range=[0, xmax]),
        yaxis=dict(range=[0, ymax])
    )
    return fig

In [ ]:
COUNTRIES = {
    "Great Britain": {"nom": "GREAT BRITAIN", "fr": (48.8,  0.5), "tgt": (50.8, -1.2), "lbl": (52.0, -4.0), "sym": ("triangle-up",    "triangle-down")},
    "Italy-North":   {"nom": "ITALY-NORTH",   "fr": (44.5,  4.5), "tgt": (44.5, 10.0), "lbl": (43.0, 11.8), "sym": ("triangle-right", "triangle-left")},
    "Spain":         {"nom": "SPAIN",          "fr": (44.0,  0.5), "tgt": (41.0, -3.0), "lbl": (40.0, -4.5), "sym": ("triangle-sw",    "triangle-ne")},
    "Switzerland":   {"nom": "SWITZERLAND",    "fr": (46.5,  4.0), "tgt": (46.8,  8.5), "lbl": (46.8, 11.8), "sym": ("triangle-right", "triangle-left")},
    "Belgium":       {"nom": "BELGIUM",        "fr": (49.5,  3.5), "tgt": (50.8,  4.3), "lbl": (52.1,  3.0), "sym": ("triangle-ne",    "triangle-sw")},
    "Germany":       {"nom": "GERMANY",        "fr": (48.5,  5.0), "tgt": (51.5,  9.5), "lbl": (53.2, 11.0), "sym": ("triangle-ne",    "triangle-sw")},
}


def _arrow_traces(fr, tgt, val_exp, val_imp, info, max_f, ecart=0.3, max_w=15):
    traces = []
    dx, dy = tgt[1] - fr[1], tgt[0] - fr[0]
    norm   = math.hypot(dx, dy)
    px, py = -dy / norm, dx / norm
    for val, color, side, sym in [
        (val_exp, "green",  1, info["sym"][0]),
        (val_imp, "red",   -1, info["sym"][1]),
    ]:
        if val <= 0:
            continue
        w    = max(1, (val / max_f) * max_w)
        lons = [fr[1] + side*ecart*px, tgt[1] + side*ecart*px]
        lats = [fr[0] + side*ecart*py, tgt[0] + side*ecart*py]
        traces.append(go.Scattergeo(lon=lons, lat=lats, mode="lines",
                                     line=dict(width=w, color=color), hoverinfo="skip"))
        tip_lon = lons[1] if color == "green" else lons[0]
        tip_lat = lats[1] if color == "green" else lats[0]
        traces.append(go.Scattergeo(lon=[tip_lon], lat=[tip_lat], mode="markers",
                                     marker=dict(symbol=sym, size=w+8, color=color), hoverinfo="skip"))
    return traces


def create_flows_map(yearly_df, year, flow_type="physical"):
    if flow_type == "physical":
        export_col, import_col, unit = "export_twh", "import_twh", "TWh"
    elif flow_type == "monetary":
        export_col, import_col, unit = "export_value_M€", "import_value_M€", "M€"
    else:
        raise ValueError("flow_type must be 'physical' or 'monetary'")

    df_year = yearly_df[yearly_df["year"] == year].copy()
    df_year["export"] = df_year[export_col]
    df_year["import"] = df_year[import_col]

    max_f = max(1, df_year["export"].max(), df_year["import"].max())
    fig   = go.Figure()
    total_exp = total_imp = 0

    for country, info in COUNTRIES.items():
        row = df_year[df_year["country"] == country]
        if row.empty:
            continue
        val_exp = row["export"].iloc[0]
        val_imp = row["import"].iloc[0]
        total_exp += val_exp
        total_imp += val_imp

        fig.add_trace(go.Scattergeo(
            lon=[info["lbl"][1], info["tgt"][1]],
            lat=[info["lbl"][0], info["tgt"][0]],
            mode="lines", line=dict(width=1, color="rgba(100,100,100,0.2)"), hoverinfo="skip"
        ))
        label = (
            f"<b>{info['nom']}</b><br>"
            f"<span style='color:green'>Exp: {val_exp:,.3f} {unit}</span><br>"
            f"<span style='color:red'>Imp: {val_imp:,.3f} {unit}</span>"
        ).replace(",", "\u202f")
        fig.add_trace(go.Scattergeo(
            lon=[info["lbl"][1]], lat=[info["lbl"][0]],
            mode="text", text=[label],
            textfont=dict(size=14, color="#0b1736"), hoverinfo="skip"
        ))
        for t in _arrow_traces(info["fr"], info["tgt"], val_exp, val_imp, info, max_f):
            fig.add_trace(t)

    solde = total_exp - total_imp
    bilan = (
        f"<b>TOTAL FRANCE {year}</b><br>"
        f"Exp: {total_exp:,.3f} {unit}<br>"
        f"Imp: {total_imp:,.3f} {unit}<br>"
        f"Net: {solde:,.3f} {unit}"
    ).replace(",", "\u202f")
    fig.add_trace(go.Scattergeo(
        lon=[-5.5], lat=[46.5], mode="text", text=[bilan],
        textfont=dict(size=15, color="black"), hoverinfo="skip"
    ))
    fig.update_layout(
        geo=dict(scope="europe", showland=True, landcolor="#e0f3f8",
                 countrycolor="white", center=dict(lon=4.0, lat=47.0), projection_scale=4),
        margin=dict(l=0, r=0, t=0, b=0),
        height=600, showlegend=False, hovermode=False, dragmode=False,
        title=f"Flux {'physiques' if flow_type=='physical' else 'monétaires'} France {year}"
    )
    return fig

## 4. Analyses par année
Modifiez la variable `YEAR` pour explorer une année spécifique, ou lancez la cellule de boucle pour les générer toutes.

In [ ]:
YEAR = 2022  # ← modifiez ici

### 4.1 Heures à forte utilisation (> 80 %)

In [ ]:
histogram_hours(hourly_report, YEAR).show()

**Lecture :** chaque barre représente une interconnexion (FR↔voisin) ; en vert les heures où FR exporte à >80 % de capacité, en rouge les heures où FR importe à >80 %. Une dominance rouge signale une congestion à l'import (la France a besoin d'importer plus que la capacité ne le permet).

### 4.2 Rente de congestion absolue

In [ ]:
histogram_congestion_rent(yearly_report, YEAR).show()

**Lecture :** rente de congestion par partenaire, décomposée selon le sens du flux. La rente mesure le revenu théorique collecté par les GRT quand les prix ne convergent pas. Plus la barre est haute, plus la frontière est économiquement congestionnée — c'est un signal d'investissement pour les régulateurs (Projets d'Intérêt Commun européens).⚠️ **Limite :** cet indicateur est calculé en valeur absolue, donc il additionne les heures où la rente est *réellement* perçue et les heures de *flux à contre-prix* (qui sont des inefficiences). La section 7 décompose ces deux phénomènes.

### 4.3 Carte de congestion (spread vs. utilisation)

In [ ]:
plot_congestion_map(yearly_report, YEAR).show()

**Lecture en 4 cadrans** (séparés par les médianes de l'échantillon) :- **Bas-gauche (vert) — Marché intégré :** capacité largement disponible, prix convergents.- **Bas-droite (bleu) — Arbitrage efficace :** capacité saturée mais prix convergents, le market coupling fait son travail.- **Haut-gauche (orange) — Barrière de marché :** spreads élevés sans saturer la capacité. Pointe vers d'autres limites : contraintes flow-based, allocations explicites (CH hors SDAC), mécanismes de capacité.- **Haut-droite (rouge) — Congestion structurelle :** vraie congestion physique justifiant un renforcement.La couleur des points indique le sens dominant du flux annuel (vert = FR exporte net, rouge = FR importe net).

### 4.4 Carte des flux physiques (TWh)

In [ ]:
create_flows_map(yearly_report, YEAR, "physical").show()

**Lecture :** flèches vertes = exports FR, rouges = imports FR, épaisseur proportionnelle au volume (TWh). Le bilan net affiché au centre est l'indicateur clé : en 2022, la France est devenue importatrice nette pour la première fois depuis 1980 (corrosion sous contrainte des réacteurs + crise gazière).

### 4.5 Carte des flux monétaires (M€)

In [ ]:
create_flows_map(yearly_report, YEAR, "monetary").show()

**Lecture :** même carte mais épaisseur proportionnelle à la **valeur économique** (M€). Une frontière peut être petite en TWh mais grosse en M€ si les prix moyens y sont élevés. En 2022 les imports valent particulièrement cher (prix moyens > 250 €/MWh).⚠️ **Note méthodologique :** `value_eur = flow × price_export`. Pour un import, `price_export` est le prix du voisin (= ce que les producteurs étrangers ont touché), pas le prix français (= ce que la France a payé). Le différentiel correspond justement à la rente de congestion.

### 4.6 Tableau récapitulatif annuel

In [ ]:
cols = [
    "country",
    "export_twh", "import_twh",
    "export_value_M€", "import_value_M€",
    "utilization_rate",
    "spread_avg_€/MWh",
    "abs_congestion_rent_M€",
]
display(
    yearly_report[yearly_report["year"] == YEAR][cols]
    .set_index("country")
    .style.format({
        "export_twh":               "{:.2f}",
        "import_twh":               "{:.2f}",
        "export_value_M€":          "{:,.0f}",
        "import_value_M€":          "{:,.0f}",
        "utilization_rate":         "{:.1%}",
        "spread_avg_€/MWh":         "{:.1f}",
        "abs_congestion_rent_M€":   "{:,.0f}",
    })
)

## 5. Évolution multi-annuelle
Graphiques agrégés sur toute la période 2014-2025.

In [ ]:
fig = go.Figure()
for country in sorted(yearly_report["country"].unique()):
    sub = yearly_report[yearly_report["country"] == country].sort_values("year")
    fig.add_trace(go.Scatter(
        x=sub["year"], y=sub["utilization_rate"],
        mode="lines+markers", name=country
    ))
fig.update_layout(
    title="Taux d'utilisation moyen par interconnexion (2014-2025)",
    xaxis_title="Année", yaxis_title="Taux d'utilisation",
    yaxis_tickformat=".0%"
)
fig.show()

**Lecture :** évolution annuelle du taux d'utilisation moyen. À surveiller : l'effet du market coupling (CWE puis Core), l'effet Brexit sur la frontière GB (allocation explicite depuis 2021), et le pic 2022.

In [ ]:
fig = go.Figure()
for country in sorted(yearly_report["country"].unique()):
    sub = yearly_report[yearly_report["country"] == country].sort_values("year")
    fig.add_trace(go.Scatter(
        x=sub["year"], y=sub["abs_congestion_rent_M€"],
        mode="lines+markers", name=country
    ))
fig.update_layout(
    title="Rente de congestion absolue par interconnexion (2014-2025)",
    xaxis_title="Année", yaxis_title="Rente (M€)"
)
fig.show()

**Lecture :** la rente totale par interconnexion sur 12 ans. Le pic 2022 est attendu et reflète les spreads extrêmes de la crise énergétique. Sur la durée, la frontière qui rapporte le plus de rente est candidate prioritaire au renforcement.

In [ ]:
yearly_report["net_twh"] = yearly_report["export_twh"] - yearly_report["import_twh"]

fig = go.Figure()
for country in sorted(yearly_report["country"].unique()):
    sub = yearly_report[yearly_report["country"] == country].sort_values("year")
    fig.add_trace(go.Scatter(
        x=sub["year"], y=sub["net_twh"],
        mode="lines+markers", name=country
    ))
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(
    title="Solde net France (Export − Import) par partenaire (2014-2025)",
    xaxis_title="Année", yaxis_title="TWh"
)
fig.show()

**Lecture :** solde net (export − import) en TWh par partenaire. Les courbes qui basculent sous zéro en 2022 = frontières où la France est devenue importatrice nette. C'est la rupture historique du bilan énergétique français.

## 6. Analyse saisonnière
Heatmaps mois × partenaire et barres par saison. Utilise la variable `YEAR` définie en section 4.

In [ ]:
MONTH_NAMES = ["Jan", "Fév", "Mar", "Avr", "Mai", "Jun",
               "Jul", "Aoû", "Sep", "Oct", "Nov", "Déc"]

def heatmap_monthly(monthly_report, year, metric="utilization_rate"):
    df = monthly_report[monthly_report["year"] == year]
    pivot = df.pivot_table(index="country", columns="month", values=metric, aggfunc="mean")

    META = {
        "utilization_rate":        ("Taux d'utilisation",     "RdYlGn", lambda v: f"{v:.0%}"),
        "spread_avg_€/MWh":       ("Spread moyen (€/MWh)",   "YlOrRd", lambda v: f"{v:.1f}"),
        "abs_congestion_rent_M€": ("Rente de congestion M€", "Reds",   lambda v: f"{v:.0f}"),
    }
    label, colorscale, fmt_fn = META.get(metric, (metric, "Blues", lambda v: f"{v:.2f}"))

    z    = pivot.values
    text = [[fmt_fn(v) if not np.isnan(v) else "" for v in row] for row in z]

    fig = go.Figure(go.Heatmap(
        z=z,
        x=[MONTH_NAMES[m - 1] for m in pivot.columns],
        y=pivot.index.tolist(),
        colorscale=colorscale,
        text=text,
        texttemplate="%{text}",
        hovertemplate="%{y} — %{x} : %{text}<extra></extra>",
    ))
    fig.update_layout(
        title=f"{label} — mois × partenaire ({year})",
        xaxis_title="Mois", yaxis_title="", height=350,
    )
    return fig


def seasonal_bars(monthly_report, year, metric="abs_congestion_rent_M€"):
    df = monthly_report[monthly_report["year"] == year]
    seasonal = (
        df.groupby(["season", "country"])[metric]
        .mean()
        .reset_index()
    )
    order = ["Hiver", "Printemps", "Été", "Automne"]
    seasonal["season"] = pd.Categorical(seasonal["season"], categories=order, ordered=True)
    seasonal = seasonal.sort_values(["season", "country"])

    fig = go.Figure()
    for country in sorted(seasonal["country"].unique()):
        sub = seasonal[seasonal["country"] == country]
        fig.add_trace(go.Bar(x=sub["season"], y=sub[metric], name=country))

    LABELS = {
        "abs_congestion_rent_M€": "Rente de congestion (M€)",
        "utilization_rate":        "Taux d'utilisation moyen",
        "spread_avg_€/MWh":       "Spread moyen (€/MWh)",
    }
    fig.update_layout(
        title=f"{LABELS.get(metric, metric)} par saison ({year})",
        xaxis_title="Saison", yaxis_title=LABELS.get(metric, metric),
        barmode="group",
    )
    return fig


# ── Affichage ────────────────────────────────────────────────────────────────

heatmap_monthly(monthly_report, YEAR, "utilization_rate").show()
heatmap_monthly(monthly_report, YEAR, "spread_avg_€/MWh").show()
seasonal_bars(monthly_report, YEAR, "abs_congestion_rent_M€").show()
seasonal_bars(monthly_report, YEAR, "utilization_rate").show()

**Lecture :**- Heatmap **utilisation** : intensité de la saturation par mois (rouge = forte). En hiver les imports concentrent la pression (chauffage électrique, faible solaire).- Heatmap **spread** : où et quand le découplage de marché est le plus fort. L'effet "exception ibérique" est visible sur ES à partir de juin 2022.- Barres **saisonnières** : concentration automne/hiver attendue de la rente, sauf cas particuliers (ES en été = solaire saturé).

## 7. Flux à contre-prix (adverse flows)Un **flux à contre-prix** est une heure où l'électricité circule du marché *cher* vers le marché *bon marché* — l'inverse de ce que le market coupling devrait produire. Mathématiquement :$$\text{flux à contre-prix} \iff \text{flow}_t > 0 \text{ et } P^{\text{import}}_t < P^{\text{export}}_t$$ce qui équivaut à `congestion_rent < 0`.**Causes possibles :**1. **Contrats long terme** non re-nominés à l'heure (historique sur frontière CH).2. **Erreurs de prévision** au moment de la nomination day-ahead, sans rééquilibrage intraday suffisant.3. **Allocations explicites** hors SDAC (CH, GB post-Brexit) — l'acheteur de capacité peut nominer à perte.4. **Contraintes flow-based** : un flux apparemment "à contre-prix" sur une frontière peut être *globalement* optimal au niveau du système couplé (Core). À garder en tête pour FR-DE, FR-BE.**Pourquoi c'est un indicateur clé :** il quantifie directement l'inefficience du market coupling, là où la rente "absolue" la masque.

In [ ]:
# ── Calcul des indicateurs de flux à contre-prix ─────────────────────────────
# On filtre les flux non significatifs (< 1 MW) pour éviter de capter du bruit numérique
MIN_FLOW = 1.0  # MW

adv = hourly_report[
    (hourly_report["flow_mw"].abs() > MIN_FLOW) &
    hourly_report["congestion_rent"].notna()
].copy()

# Flag adverse flow
adv["adverse"] = adv["congestion_rent"] < 0

# Direction (export France / import France)
adv["direction"] = np.where(
    adv["from_country"] == "France", "Export FR", "Import FR"
)

# Coût d'inefficience (en valeur absolue, uniquement quand adverse)
adv["adverse_cost_eur"] = np.where(adv["adverse"], -adv["congestion_rent"], 0.0)

# Agrégation annuelle par partenaire + direction
adverse_yearly = (
    adv.groupby(["year", "partner", "direction"])
       .agg(
           total_hours=("adverse", "size"),
           adverse_hours=("adverse", "sum"),
           adverse_cost_eur=("adverse_cost_eur", "sum"),
       )
       .reset_index()
)
adverse_yearly["adverse_cost_M_eur"] = adverse_yearly["adverse_cost_eur"] / 1e6
adverse_yearly["adverse_share_pct"]  = 100 * adverse_yearly["adverse_hours"] / adverse_yearly["total_hours"]
adverse_yearly = adverse_yearly.rename(columns={"partner": "country"})

# Total France toutes frontières
total_adv = (
    adv.groupby("year")
       .agg(total_hours=("adverse", "size"),
            adverse_hours=("adverse", "sum"),
            adverse_cost_eur=("adverse_cost_eur", "sum"))
)
total_adv["adverse_cost_M_eur"]  = total_adv["adverse_cost_eur"] / 1e6
total_adv["adverse_share_pct"]   = 100 * total_adv["adverse_hours"] / total_adv["total_hours"]

print("Total flux à contre-prix toutes frontières confondues :")
print(total_adv[["adverse_share_pct", "adverse_cost_M_eur"]].round(1))

### 7.1 Part des heures à contre-prix par frontière

In [ ]:
d = adverse_yearly[adverse_yearly["year"] == YEAR]

fig = go.Figure()
for direction, color in [("Export FR", "green"), ("Import FR", "red")]:
    sub = d[d["direction"] == direction]
    fig.add_trace(go.Bar(
        x=sub["country"], y=sub["adverse_share_pct"],
        name=direction, marker_color=color,
        text=sub["adverse_share_pct"].round(1), textposition="outside"
    ))
fig.update_layout(
    title=f"Part des heures à contre-prix par frontière ({YEAR})",
    xaxis_title="Partenaire", yaxis_title="% des heures à contre-prix",
    barmode="group"
)
fig.show()

**Lecture :** part des heures où le flux est à contre-prix (mode `barmode="group"` pour séparer export/import). Les barres élevées signalent une **inefficience du market coupling** sur cette frontière.**Repères typiques (littérature ACER) :**- < 2 % : marché bien couplé.- 2–5 % : inefficiences résiduelles (bruit de prévision, intraday).- \> 5 % : problème structurel (frontière non-SDAC, contraintes flow-based mal calibrées, contrats long terme).La Suisse devrait sortir en tête (frontière hors SDAC, allocation explicite). La GB est intéressante à comparer pré/post-Brexit (2021).

### 7.2 Coût économique des flux à contre-prix

In [ ]:
d = adverse_yearly[adverse_yearly["year"] == YEAR]

fig = go.Figure()
for direction, color in [("Export FR", "green"), ("Import FR", "red")]:
    sub = d[d["direction"] == direction]
    fig.add_trace(go.Bar(
        x=sub["country"], y=sub["adverse_cost_M_eur"],
        name=direction, marker_color=color
    ))
fig.update_layout(
    title=f"Coût des flux à contre-prix par frontière ({YEAR})",
    xaxis_title="Partenaire", yaxis_title="Coût (M€)",
    barmode="stack"
)
fig.show()

**Lecture :** somme des |rentes négatives| par frontière — ce qu'on peut interpréter comme la **valeur économique perdue** à cause de flux mal-orientés (perte d'efficience allocative).**À comparer avec la rente totale (section 4.2)** pour relativiser. Si le coût des contre-prix représente 10 % de la rente totale d'une frontière, c'est un problème majeur ; si c'est 1 %, c'est marginal.

### 7.3 Évolution multi-annuelle du taux de contre-prix

In [ ]:
# Agrégation toutes directions confondues pour visualiser la tendance
adverse_all = (
    adv.groupby(["year", "partner"])
       .agg(total_hours=("adverse", "size"), adverse_hours=("adverse", "sum"))
       .reset_index()
       .rename(columns={"partner": "country"})
)
adverse_all["adverse_share_pct"] = 100 * adverse_all["adverse_hours"] / adverse_all["total_hours"]

fig = go.Figure()
for country in sorted(adverse_all["country"].unique()):
    sub = adverse_all[adverse_all["country"] == country].sort_values("year")
    fig.add_trace(go.Scatter(
        x=sub["year"], y=sub["adverse_share_pct"],
        mode="lines+markers", name=country
    ))

# Repères temporels marquants
fig.add_vline(x=2021, line_dash="dash", line_color="gray",
              annotation_text="Brexit (FR-GB hors SDAC)", annotation_position="top")
fig.add_vline(x=2022.5, line_dash="dash", line_color="gray",
              annotation_text="Core flow-based", annotation_position="bottom")

fig.update_layout(
    title="Évolution du taux de contre-prix par frontière (2014-2025)",
    xaxis_title="Année", yaxis_title="% des heures à contre-prix"
)
fig.show()

**Lecture :** évolution sur 12 ans du taux de contre-prix par frontière. Deux ruptures à surveiller :1. **Brexit 2021 sur FR-GB** : passage du SDAC à l'allocation explicite. On attend une hausse nette du taux de contre-prix.2. **Lancement Core flow-based en juin 2022** : extension du flow-based à 13 zones (vs 5 en CWE). Effet ambigu sur les frontières FR-DE et FR-BE — à observer.Une tendance baissière sur 12 ans pour une frontière donnée = amélioration du market coupling sur cette interconnexion.

## 8. Notes méthodologiques**Choix et limites de la méthodologie utilisée :**1. **Spread moyen pondéré par les flux.** `spread_avg_€/MWh = rente_absolue_totale / flux_total`. C'est une moyenne pondérée du |spread| par le volume — donc les heures de fort flux pèsent plus. Une moyenne arithmétique horaire donnerait un chiffre différent (en général plus bas).2. **`import_value_M€` ≠ "ce que la France paie".** La valeur est calculée comme `flow × price_export` (prix du pays exportateur). Pour un import vers la France, c'est le prix du voisin qui s'applique — soit ce que les producteurs étrangers encaissent, et non ce que les consommateurs français déboursent. Le différentiel est précisément la rente de congestion.3. **Capacités physiques vs commerciales.** On utilise les capacités "Day-ahead Total Transfer Capacities" (NTC). Sur les frontières flow-based (DE, BE depuis 2015 ; étendu via Core en 2022), la capacité bilatérale n'est plus une grandeur exacte — c'est une approximation. Les taux d'utilisation > 100 % observés ponctuellement reflètent ce phénomène.4. **Flux physiques vs commerciaux.** L'analyse repose sur les flux **physiques** (mesurés sur les lignes). Les flux **commerciaux** (nominations day-ahead) peuvent différer significativement à cause des *loop flows* (ex : éolien allemand transitant par la France pour rejoindre l'Italie). Une analyse rigoureuse devrait croiser les deux — non fait ici faute de données scheduled exchanges.5. **Taux de change GBP→EUR.** Conversion linéaire au taux moyen annuel BCE. Sur 2022, l'amplitude intra-annuelle (≈ 8 %) peut introduire un biais de l'ordre du %.6. **Définition du seuil pour les flux à contre-prix.** Filtré sur `|flow| > 1 MW` pour éviter les artefacts numériques. Définition stricte : `congestion_rent < 0`. Une définition plus stricte exigerait `|spread| > seuil_de_signification` (typiquement 1 €/MWh).